In [ ]:
import sys
import subprocess
import os
import re
from importlib.metadata import version, PackageNotFoundError

# tomllib is native to Python 3.11+
try:
    import tomllib
except ImportError:
    print("[!] CRITICAL ERROR: tomllib not found. Python 3.11+ is strictly required.")
    sys.exit(1)

def check_environment(toml_path="pyproject.toml", auto_install=False):
    """
    Checks Python version and utilizes importlib.metadata to verify dependencies safely
    against the pyproject.toml specifications.
    """
    print("\n" + "<"*30 + ">"*30)
    print("   SYSTEM ENVIRONMENT & DEPENDENCY CHECK")
    print("<"*30 + ">"*30 + "\n")

    # 1. CHECK PYTHON VERSION
    current_py = sys.version_info
    if current_py < (3, 11) or current_py >= (3, 13):
        print(f"[!] WARNING: Incompatible Python version detected: {sys.version.split()[0]}")
        print("    This pipeline is optimized for Python 3.11 or 3.12.")
        print("    Python 3.13+ may cause compilation errors with NumPy 1.x.")

        if not auto_install:
            try:
                input("    Press Enter to attempt to continue, or Ctrl+C to exit...")
            except KeyboardInterrupt:
                print("\n\n[!] Exiting: Environment check cancelled by user.")
                return False

    # 2. CHECK PYPROJECT FILE
    if not os.path.exists(toml_path):
        print(f"\n[!] ERROR: '{toml_path}' not found.")
        print("    Ensure you are running this script from the project root.")
        return False

    try:
        with open(toml_path, "rb") as f:
            project_data = tomllib.load(f)
        required_lines = project_data.get("project", {}).get("dependencies", [])
    except Exception as e:
        print(f"\n[!] ERROR: Failed to parse '{toml_path}'.")
        print(f"    Details: {e}")
        return False

    if not required_lines:
        print(f"[!] Notice: No dependencies found in '{toml_path}'. Skipping dependency check.")
        return True

    # 3. VERIFY INDIVIDUAL DEPENDENCIES
    to_install = []
    print(f"Analyzing {len(required_lines)} dependencies...\n")

    for req in required_lines:
        # Extract base package name (e.g., "pandas>=2.0.0" -> "pandas")
        match = re.split(r'[=><~]', req, 1)
        pkg_name = match[0].strip()

        try:
            installed_ver = version(pkg_name)
        except PackageNotFoundError:
            print(f"    [!] Missing completely: {req}")
            to_install.append(req)
        except Exception as e:
            print(f"    [!] Error checking {pkg_name}: {e}")
            to_install.append(req)

    # 3.5 TRAP CHECK: COMMUNITY VS PYTHON-LOUVAIN
    try:
        import community
        if not hasattr(community, 'best_partition'):
            print("\n    [!] NAME COLLISION DETECTED: The generic 'community' package is shadowing 'python-louvain'.")
            print("        Uninstalling the conflicting package now...")
            subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "community"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            if "python-louvain" not in to_install:
                to_install.append("python-louvain")
    except ImportError:
        pass # Handled by the standard requirements check

    # 4. TARGETED PACKAGE INSTALLATION
    if to_install:
        print(f"\n[!] Found {len(to_install)} missing or unverified dependencies.")

        if not auto_install:
            try:
                choice = input("    Run package installation (pip install -e .) now? (y/n): ").strip().lower()
                if choice != 'y':
                    print("\n[!] Exiting: User declined to install required dependencies.")
                    return False
            except KeyboardInterrupt:
                print("\n\n[!] Exiting: Installation prompt cancelled by user.")
                return False
        else:
            print("    Auto-install flag detected. Proceeding with installation.")

        print("\nStarting package installation...")
        try:
            # Install the local directory as an editable package, resolving all dependencies
            result = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-e", "."],
                stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
            )
            if result.returncode != 0:
                print(f"\n[!] CRITICAL ERROR: Package installation failed.")
                print("--- PIP ERROR LOG ---")
                print(result.stderr.strip())
                print("---------------------")
                return False
        except Exception as e:
            print(f"\n[!] CRITICAL ERROR: Unexpected system failure installing package.")
            print(f"    Details: {e}")
            return False
        print("\n[+] Package and dependencies installed successfully.")

    print("[+] Environment and dependencies are fully verified.")
    return True

def provision_kaleido_dependencies():
    """Installs required OS-level shared libraries and the headless Chrome binary."""
    if not sys.platform.startswith('linux'):
        print("\n[!] Non-Linux OS detected. Skipping OS-level Kaleido dependencies.")
        return

    print("\nProvisioning system dependencies for headless Chrome rendering...")

    # 1. Install Linux shared libraries
    try:
        env = os.environ.copy()
        env["DEBIAN_FRONTEND"] = "noninteractive"

        subprocess.run(["apt-get", "update"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env)
        libs = [
            "libnss3", "libatk-bridge2.0-0", "libxcomposite1",
            "libxdamage1", "libxrandr2", "libgbm1", "libasound2"
        ]
        subprocess.run(["apt-get", "install", "-y"] + libs, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env)
        print("    [+] System libraries provisioned.")
    except subprocess.CalledProcessError as e:
        print(f"    [!] Failed to install system libraries: {e}")
        print("        Note: If not running as root/Colab, this requires sudo.")

    # 2. Install Headless Chrome via Plotly
    try:
        print("    Downloading headless Chromium binary (auto-confirming prompt)...")

        subprocess.run(
            ["plotly_get_chrome"],
            check=True,
            input="y\n",
            text=True
        )
        print("    [+] Headless Chromium binary installed.")
    except subprocess.CalledProcessError:
        print(f"    [!] Warning: Failed to install headless Chromium.")
        print("        Static HTML files will be generated as fallbacks.")
    except FileNotFoundError:
        print("    [!] Error: 'plotly_get_chrome' command not found.")
        print("        Ensure 'plotly >= 6.1.1' is installed via pyproject.toml first.")

def main():
    auto_mode = "--auto" in sys.argv
    try:
        success = check_environment(auto_install=auto_mode)
        if not success:
            sys.exit(1)
    except KeyboardInterrupt:
        print("\n\n[!] Exiting: Script manually interrupted by user.")
        sys.exit(130)

    provision_kaleido_dependencies()

if __name__ == "__main__":
    main()